In [1]:
from IPython.display import Markdown, display

import gcsfs
import pandas as pd

from update_vars import PROCESSED_GCS, DIGEST_DICT, abbrev_month

In [2]:
operator_summary_url = f"{PROCESSED_GCS}{DIGEST_DICT.operator_summary}_{abbrev_month}.parquet"

operator_df = pd.read_parquet(
    operator_summary_url,
    filesystem = gcsfs.GCSFileSystem(),
    filters=[
        ("Day Type", "==", "Weekday"),
        #("Analysis Name", "==", analysis_name)
    ],
).drop_duplicates(
    subset = ["Analysis Name", "Date"]
).reset_index(drop=True)

bridge contains schedule_gtfs_dataset_name to analysis_name to ntd_id
* fanout can occur if multiple feeds are combined to same analysis name
   * Redding Remix, Schedule, Flex are all Redding name
   * they share the same ntd_id
* how should query be set up to remove the deduping needed?
   * can just download dim_annual_agency_information with the most recent
   * make sure the join here works to attach NTD profile information
   * crosswalk would keep analysis_name - ntd_id and join with dim_annual_agency_deduped_and_downloaded

In [3]:
ntd_columns = [
    "ntd_id",
    "service_area_sq_miles", "service_area_pop", "primary_uza_name",
    "agency_name", 
    # dim table's agency_name is not source_agency that is presented in ridership reports
    # that agency name is the name that comes in the report, confusing!
    "reporter_type", "reporting_module"]

ntd_url = f"{PROCESSED_GCS}dim_annual_agency_information.parquet"
ntd_profile_df = pd.read_parquet(
    ntd_url, 
    filesystem = gcsfs.GCSFileSystem(),
    columns = ntd_columns
).rename(columns = {"ntd_id": "ntd_id_2022"})

In [4]:
crosswalk = pd.read_parquet(
    f"{PROCESSED_GCS}crosswalk_{abbrev_month}.parquet",
    filesystem = gcsfs.GCSFileSystem(),
    columns = ["analysis_name", "ntd_id_2022", "county_name", "caltrans_district"]
).rename(
    columns = {"analysis_name": "Analysis Name"}
).drop_duplicates().reset_index(drop=True)
# dedupe here because multiple schedule_gtfs_dataset_names can share same analysis_name


In [5]:
m1 = pd.merge(
    operator_df,
    crosswalk,
    on = "Analysis Name",
    how = "outer",
    indicator = True
)
m1._merge.value_counts()

_merge
both          2181
right_only      12
left_only        0
Name: count, dtype: int64

In [6]:
m2 = pd.merge(
    m1,
    ntd_profile_df,
    on = "ntd_id_2022",
    how = "left",
    indicator="ntd_merge"
)

# don't care about right_only, we know lots of NTD IDs for rest of US is in that df
m2.ntd_merge.value_counts()

ntd_merge
both          2125
left_only       68
right_only       0
Name: count, dtype: int64

In [7]:
m2[m2.ntd_merge=="left_only"]["Analysis Name"].value_counts()

Analysis Name
Calaveras Transit Agency    14
City of Elk Grove           14
City of Lawndale            14
Tehama County               14
Foothill Transit            12
Name: count, dtype: int64

In [8]:
find_me = ["City of Elk Grove", "City of Lawndale", "Foothill Transit"]
operator_df[operator_df["Analysis Name"].isin(find_me)].head()

,Date,Analysis Name,Caltrans District,VP Name,TU Name,N Trips,Day Type,Daily Trips,Ttl Service Hours,N Routes,...,N Stops,Daily Arrivals,VP Messages Per Minute,N VP Trips,Daily VP Trips,Pct VP Trips,TU Messages Per Minute,N TU Trips,Daily TU Trips,Pct TU Trips
33,2025-09-01,City of Elk Grove,03 - Marysville / Sacramento,None,None,0,Weekday,0.0,0.0,0,...,0,0.0,NaN,0,0.0,NaN,NaN,0,0.0,NaN
48,2026-06-01,Foothill Transit,07 - Los Angeles / Ventura,None,None,0,Weekday,0.0,0.0,0,...,0,0.0,NaN,0,0.0,NaN,NaN,0,0.0,NaN
132,2026-05-01,City of Elk Grove,03 - Marysville / Sacramento,None,None,0,Weekday,0.0,0.0,0,...,0,0.0,NaN,0,0.0,NaN,NaN,0,0.0,NaN
136,2026-02-01,City of Elk Grove,03 - Marysville / Sacramento,None,None,0,Weekday,0.0,0.0,0,...,0,0.0,NaN,0,0.0,NaN,NaN,0,0.0,NaN
154,2025-11-01,City of Elk Grove,03 - Marysville / Sacramento,None,None,0,Weekday,0.0,0.0,0,...,0,0.0,NaN,0,0.0,NaN,NaN,0,0.0,NaN


In [9]:
m2[m2["Analysis Name"].isin(find_me)].ntd_merge.value_counts()

ntd_merge
left_only     40
both          26
right_only     0
Name: count, dtype: int64

In [10]:
# this is ok, we have a record that does get merged correctly
m2[m2["Analysis Name"].isin(find_me)][["Analysis Name", "ntd_id_2022", "ntd_merge"]
    ].drop_duplicates().sort_values("Analysis Name")

,Analysis Name,ntd_id_2022,ntd_merge
389,City of Elk Grove,90019,both
390,City of Elk Grove,90205,left_only
542,City of Lawndale,90280,left_only
1047,Foothill Transit,90146,both
1048,Foothill Transit,90264,left_only


Foothill Transit latest ntd_id in 2023 is "90146"
* But the one in crosswalk has organization_name City of Duarte, and that ntd_id_2022 of "90264" is not found anymore, which is correct
* why is the bridge table bringing in City of Duarte only, where is Foothill Schedule? it should have its own schedule_gtfs_dataset_name, and we should be seeing 2 records
  

In [11]:
analysis_name = "Alameda-Contra Costa Transit District"
test = m2[m2["Analysis Name"] == analysis_name]

In [12]:
service_area = test.service_area_sq_miles.values[0]
service_pop = test.service_area_pop.values[0]
service_area, service_pop

(364, 1586454)

In [13]:
display(
    Markdown(
        f"""{analysis_name} is headquartered in <b>{test.county_name.values[0]}</b> 
        County in the Urbanized Area of <b>{test.primary_uza_name.values[0]}</b>.<br>
        This operator provides <b>{service_area}</b> square miles of public transit service, which has a 
        service population of <b>{service_pop}</b>.<br>
        This organization is a {test.reporter_type.values[0]}.<br>
        <b>Data Source</b>: <a href="https://www.transit.dot.gov/ntd/data-product/2022-annual-database-agency-information">National Transit Database</a> Annual Agency Information.
        """
    )
)

Alameda-Contra Costa Transit District is headquartered in <b>Alameda</b> 
        County in the Urbanized Area of <b>San Francisco--Oakland, CA</b>.<br>
        This operator provides <b>364</b> square miles of public transit service, which has a 
        service population of <b>1586454</b>.<br>
        This organization is a Full Reporter.<br>
        <b>Data Source</b>: <a href="https://www.transit.dot.gov/ntd/data-product/2022-annual-database-agency-information">National Transit Database</a> Annual Agency Information.
        